# 03 Longitudinal Feedback Simulation

Simulate 5 retraining iterations and compare diversity decay between SVD (CF) and TF-IDF (CBF).

## 1) Setup

In [ ]:
import gc
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline

from surprise import Dataset, Reader, KNNBasic, SVD

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = ROOT / 'data'
FIG_DIR = ROOT / 'outputs' / 'figures'
RES_DIR = ROOT / 'outputs' / 'results'
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

import sys
sys.path.append(str(ROOT / 'src'))
from utils import load_dataset, ensure_output_dirs
ensure_output_dirs(ROOT)

RANDOM_STATE = 42
TOP_K = 10


## 2) Config

In [ ]:
DATASET_NAME = 'ml-100k'
ITERATIONS = 5
TOP_K = 5


## 3) Data Load

In [ ]:
ratings, item_features = load_dataset(
    dataset_name=DATASET_NAME,
    data_root=DATA_ROOT,
    lastfm_mode='1-5',
    apply_k_core=True,
    min_user_interactions=10,
    min_item_interactions=10,
)
ratings = ratings.dropna().reset_index(drop=True)
base_train, holdout = train_test_split(ratings, test_size=0.2, random_state=RANDOM_STATE)
base_train = base_train.reset_index(drop=True)
holdout = holdout.reset_index(drop=True)


## 4) Helpers

In [ ]:
def aggregate_diversity(recs):
    items = set([i for lst in recs.values() for i in lst])
    return len(items)


def compute_arp_simple(recs, pop_map):
    vals = [pop_map.get(i, 0.0) for lst in recs.values() for i in lst]
    return float(np.mean(vals)) if vals else 0.0


def simulate_clicks(recs, relevance_scores, pop_map, rng, rel_weight=0.7, pop_weight=0.3):
    rows = []
    pop_vals = np.array(list(pop_map.values()), dtype=float)
    p_min, p_max = float(pop_vals.min()) if len(pop_vals) else 0.0, float(pop_vals.max()) if len(pop_vals) else 1.0

    for u, items in recs.items():
        for it in items:
            rel = relevance_scores.get((u, it), 0.5)
            if p_max > p_min:
                p_norm = (pop_map.get(it, p_min) - p_min) / (p_max - p_min)
            else:
                p_norm = 0.0
            click_prob = np.clip(rel_weight * rel + pop_weight * p_norm, 0.0, 1.0)
            click = rng.binomial(1, click_prob)
            if click == 1:
                rows.append((u, it, 5.0))
    return pd.DataFrame(rows, columns=['user_id', 'item_id', 'rating'])


def train_svd_and_recommend(train_df, users, all_items, k=5):
    reader = Reader(rating_scale=(float(train_df['rating'].min()), float(train_df['rating'].max())))
    data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
    trainset = data.build_full_trainset()
    svd = SVD(random_state=RANDOM_STATE)
    svd.fit(trainset)

    seen = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
    recs = {}
    rel_scores = {}
    for u in tqdm(users, desc='SVD recs'):
        cands = [i for i in all_items if i not in seen.get(u, set())]
        preds = [(it, svd.predict(u, it).est) for it in cands]
        preds.sort(key=lambda x: x[1], reverse=True)
        top = preds[:k]
        recs[u] = [it for it, _ in top]
        for it, score in top:
            rel_scores[(u, it)] = float((score - train_df['rating'].min()) / (train_df['rating'].max() - train_df['rating'].min() + 1e-9))

    del svd
    gc.collect()
    return recs, rel_scores


def train_tfidf_and_recommend(train_df, item_features, users, all_items, k=5):
    item_features_local = item_features[item_features['item_id'].isin(all_items)].drop_duplicates('item_id').reset_index(drop=True)
    tfidf = TfidfVectorizer(min_df=1, max_features=30000)
    mat = tfidf.fit_transform(item_features_local['metadata_text'])
    sim = cosine_similarity(mat, dense_output=False)
    idx = {it: i for i, it in enumerate(item_features_local['item_id'])}
    hist = train_df.groupby('user_id')['item_id'].apply(list).to_dict()

    recs = {}
    rel_scores = {}
    for u in tqdm(users, desc='TF-IDF recs'):
        seen = set(hist.get(u, []))
        seen_idx = [idx[i] for i in seen if i in idx]
        if not seen_idx:
            recs[u] = all_items[:k]
            for it in recs[u]:
                rel_scores[(u, it)] = 0.5
            continue

        prof = sim[seen_idx].mean(axis=0).A1
        cand = []
        for it in all_items:
            if it in seen or it not in idx:
                continue
            score = prof[idx[it]]
            cand.append((it, score))
        cand.sort(key=lambda x: x[1], reverse=True)
        top = cand[:k]
        recs[u] = [it for it, _ in top]
        for it, score in top:
            rel_scores[(u, it)] = float(score)

    del sim, mat, tfidf
    gc.collect()
    return recs, rel_scores


## 5) Simulation Loop (5 Iterations)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
users = sorted(base_train['user_id'].unique().tolist())

svd_train = base_train.copy()
tfidf_train = base_train.copy()

records = []

for t in range(1, ITERATIONS + 1):
    print(f'Iteration {t}/{ITERATIONS}')

    all_items_svd = sorted(svd_train['item_id'].unique().tolist())
    pop_svd = svd_train['item_id'].value_counts().to_dict()
    recs_svd, rel_svd = train_svd_and_recommend(svd_train, users, all_items_svd, k=TOP_K)
    clicks_svd = simulate_clicks(recs_svd, rel_svd, pop_svd, rng, rel_weight=0.7, pop_weight=0.3)

    div_svd = aggregate_diversity(recs_svd)
    arp_svd = compute_arp_simple(recs_svd, pop_svd)
    records.append({'iteration': t, 'model': 'SVD', 'aggregate_diversity': div_svd, 'arp': arp_svd})

    if len(clicks_svd) > 0:
        svd_train = pd.concat([svd_train, clicks_svd], ignore_index=True)
        svd_train = svd_train.drop_duplicates(subset=['user_id', 'item_id'], keep='last')

    del recs_svd, rel_svd, clicks_svd
    gc.collect()

    all_items_t = sorted(tfidf_train['item_id'].unique().tolist())
    pop_t = tfidf_train['item_id'].value_counts().to_dict()
    recs_t, rel_t = train_tfidf_and_recommend(tfidf_train, item_features, users, all_items_t, k=TOP_K)
    clicks_t = simulate_clicks(recs_t, rel_t, pop_t, rng, rel_weight=0.7, pop_weight=0.3)

    div_t = aggregate_diversity(recs_t)
    arp_t = compute_arp_simple(recs_t, pop_t)
    records.append({'iteration': t, 'model': 'TF-IDF', 'aggregate_diversity': div_t, 'arp': arp_t})

    if len(clicks_t) > 0:
        tfidf_train = pd.concat([tfidf_train, clicks_t], ignore_index=True)
        tfidf_train = tfidf_train.drop_duplicates(subset=['user_id', 'item_id'], keep='last')

    del recs_t, rel_t, clicks_t
    gc.collect()

sim_df = pd.DataFrame(records)
sim_df


## 6) Visualization: Diversity Decay

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=sim_df, x='iteration', y='aggregate_diversity', hue='model', marker='o')
plt.title(f'Diversity Decay over Time ({DATASET_NAME})')
plt.ylabel('Aggregate Diversity (Unique Recommended Items)')
plt.xlabel('Timestep')
plt.xticks(range(1, ITERATIONS + 1))
plt.tight_layout()
out_path = FIG_DIR / f'feedback_diversity_decay_{DATASET_NAME}.png'
plt.savefig(out_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved figure: {out_path}')


## 7) Save Results

In [ ]:
out_csv = RES_DIR / f'feedback_simulation_metrics_{DATASET_NAME}.csv'
sim_df.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')
